In [2]:
import requests
import pandas as pd
import json
def coleta_nomes_cidades():

    url_ibge = 'https://servicodados.ibge.gov.br/api/v1/localidades/municipios'
    lista_municipios = []
    r = requests.get(url_ibge)
    if r.status_code == 200:
        lista = r.json()
    else:
        print(f"Erro na requisição: {r.status_code}")
    try:
        for posicao in lista:
            lista_municipios.append({'Cidade' : posicao['nome'],'Estado' : posicao['microrregiao']['mesorregiao']['UF']['nome']})
        return lista_municipios
    except(TypeError):
        print( f'Erro: {TypeError}')
    return lista_municipios

In [3]:
enderecos = coleta_nomes_cidades()


Erro: <class 'TypeError'>


In [4]:
enderecos = pd.DataFrame(enderecos)
enderecos

,Cidade,Estado
0,Alta Floresta D'Oeste,Rondônia
1,Ariquemes,Rondônia
2,Cabixi,Rondônia
3,Cacoal,Rondônia
4,Cerejeiras,Rondônia
...,...,...
5194,Arenápolis,Mato Grosso
5195,Aripuanã,Mato Grosso
5196,Barão de Melgaço,Mato Grosso
5197,Barra do Bugres,Mato Grosso


In [11]:
mask_mg = enderecos['Estado'] == 'Minas Gerais'
cidades_mg = enderecos[mask_mg]
cidades_mg.head()

,Cidade,Estado
2244,Abadia dos Dourados,Minas Gerais
2245,Abaeté,Minas Gerais
2246,Abre Campo,Minas Gerais
2247,Acaiaca,Minas Gerais
2248,Açucena,Minas Gerais


In [18]:
from geopy.geocoders import Nominatim
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Crie o objeto UMA VEZ
geolocator = Nominatim(user_agent="meu_app_python_geocoding_123", timeout=10)

def obter_lat_lon(endereco):
    try:
        localizacao = geolocator.geocode(endereco)
        if localizacao:
            return localizacao.latitude, localizacao.longitude
        else:
            return None, None
    except Exception as e:
        # É bom saber se algo deu errado
        print(f"Erro ao geocodificar '{endereco}': {e}")
        return None, None

In [22]:
enderecos_completos = [f"{row['Cidade']}, {row['Estado']}, Brasil" for index, row in cidades_mg.iterrows()]
enderecos_completos[0]

'Abadia dos Dourados, Minas Gerais, Brasil'

In [ ]:
with ThreadPoolExecutor(max_workers=20) as executor:
    # Cria as tarefas
    futures = [executor.submit(obter_lat_lon_seguro, endereco) for endereco in enderecos]



TypeError: 'type' object does not support the context manager protocol

In [17]:
# Supondo que obter_lat_lon(retorne uma tupla (lat, lon))
cidades_mg[['lat', 'lon']] = cidades_mg.apply(
    lambda row: obter_lat_lon(f"{row['Cidade']}, {row['Estado']}, Brasil"),
    axis=1, result_type='expand'
)


KeyboardInterrupt: 

In [19]:
enderecos

,Cidade,Estado
0,Alta Floresta D'Oeste,Rondônia
1,Ariquemes,Rondônia
2,Cabixi,Rondônia
3,Cacoal,Rondônia
4,Cerejeiras,Rondônia
...,...,...
5194,Arenápolis,Mato Grosso
5195,Aripuanã,Mato Grosso
5196,Barão de Melgaço,Mato Grosso
5197,Barra do Bugres,Mato Grosso


In [14]:
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="meu_app")

# Nome do município, estado e país
municipio = "Uberlândia, Minas Gerais, Brasil"

localizacao = geolocator.geocode(municipio)

if localizacao:
    print(f"Latitude: {localizacao.latitude}")
    print(f"Longitude: {localizacao.longitude}")
else:
    print("Local não encontrado.")


Latitude: -18.9188041
Longitude: -48.2767837
